# DWN on JSC — training notebook (Kaggle)

Phase 1b of the dwn-fpga project. Produces a trained n=6 DWN checkpoint on JSC for the
exporter to consume.

**Why this runs on Kaggle and not locally:** upstream `torch_dwn` has no CPU path —
`lut_layer.py` raises `EFDFunction CPU not Implemented` in both forward and backward, and only
a CUDA extension ships. See project-brief.md §12 risk #7.

## Before you run anything

In the Kaggle settings panel on the right:

1. **Accelerator → GPU** (P100 or T4 x2, either is fine)
2. **Internet → On** — *off by default*. Without it the git clone and the OpenML fetch both fail.

## What you get out

`/kaggle/working/dwn_jsc_checkpoint.pt` — download it and commit it (or its metadata) to the repo.
It contains the model `state_dict`, the full config, **and the thermometer thresholds**, which are
not part of the state_dict but are absolutely part of the model — the hardware encoder needs them.

In [ ]:
# ---- environment check: fail loudly and early ----
import subprocess, sys, torch

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu       :', torch.cuda.get_device_name(0))
    print('cuda (torch built against):', torch.version.cuda)
else:
    raise SystemExit('No GPU. Set Accelerator -> GPU in the settings panel. '
                     'DWN training cannot run on CPU.')

print()
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
print('If the nvcc CUDA version and the torch CUDA version differ a lot, the extension')
print('build in the next cell is where it will show up.')

In [ ]:
# ---- clone upstream DWN at the pinned commit and build the CUDA extension ----
# Pin matches third_party/DWN in the repo. Do not float this to main: the exporter is
# built against whatever checkpoint format this commit produces (CLAUDE.md).
PINNED_COMMIT = '9f887a0b4bd84dabf6d8c9ae35368ab2a7e0e3c0'

!rm -rf /kaggle/working/DWN
!git clone --quiet https://github.com/alanbacellar/DWN.git /kaggle/working/DWN
!cd /kaggle/working/DWN && git checkout --quiet {PINNED_COMMIT} && git log -1 --format='pinned at %h %ad %s'

# This compiles efd_cuda_kernel.cu with nvcc. Expect 2-5 minutes. It is the slowest and
# most fragile step in the notebook.
!cd /kaggle/working/DWN && pip install --quiet . 2>&1 | tail -20

In [ ]:
# ---- VERIFY the extension actually built ----
# This cell exists because the failure is otherwise silent. lut_layer.py does
#     if torch.cuda.is_available(): import efd_cuda
# so a failed build produces no error at install time -- you would instead get a bare
# NameError at the first forward pass, long after the real cause.
import torch, torch_dwn as dwn

try:
    import efd_cuda
    print('efd_cuda imported OK')
except ImportError as e:
    raise SystemExit(
        'efd_cuda failed to import -- the CUDA extension did not build.\n'
        'Re-run the install cell without `--quiet` and read the nvcc errors.\n'
        f'Original error: {e}'
    )

# tiny end-to-end forward+backward, so we find out here rather than 200 lines later
_probe = torch.nn.Sequential(dwn.LUTLayer(12, 6, n=6), dwn.GroupSum(k=2, tau=1.0)).cuda()
_x = (torch.rand(4, 12, device='cuda') > 0.5).float()
_out = _probe(_x)
_out.sum().backward()
print('forward + backward OK, output shape', tuple(_out.shape))
del _probe, _x, _out

In [ ]:
# ---- CONFIG ----
# One dict drives the whole flow. This mirrors the Phase 2 requirement in docs/dse-plan.md §1:
# a sweep point is a config, not a code edit. Keep it that way.
CONFIG = {
    # --- encoding (Group A) ---
    'thermometer': 'distributive',   # 'plain' | 'gaussian' | 'distributive'
    'thermometer_bits': 4,           # bits per feature; 16 features -> 16*bits input width

    # --- architecture (Group A) ---
    'n': 6,                          # fixed at 6 for Phase 1 (CLAUDE.md)
    'layers': [300, 100],            # LUT nodes per layer -- the primary area dial
    'first_layer_mapping': 'learnable',

    # --- reduction ---
    # GroupSum is popcount-style and is all upstream ships. Learnable Reduction is deferred
    # until we know what the popcount costs in LUTs -- docs/dse-plan.md, Group A.
    'num_classes': 5,
    'tau': 1 / 0.3,

    # --- training (does not affect hardware) ---
    'epochs': 30,
    'batch_size': 256,
    'lr': 1e-2,
    'lr_step': 14,
    'lr_gamma': 0.1,
    'seed': 20260802,
}

# Predicted core area, from the docs/dse-plan.md §5 model: one node == one LUT6.
# Excludes the thermometer encoder (brief §12 risk #3 -- up to 3.2x more).
core_luts = sum(CONFIG['layers'])
print('config:', CONFIG)
print()
print('input bits         :', 16 * CONFIG['thermometer_bits'])
print('predicted core LUTs:', core_luts, '(encoder NOT included)')
print('pct of xc7a35t     : {:.1f}%'.format(100 * core_luts / 20800))

In [ ]:
# ---- load JSC ----
# hls4ml_lhc_jets_hlf: 16 high-level jet-substructure features, 5 classes (g, q, w, z, t).
# This is the standard low-latency FPGA-ML benchmark -- see project-brief.md §7 and §8.
import numpy as np, torch
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

data = fetch_openml('hls4ml_lhc_jets_hlf', version=1, as_frame=True)
X = data.data.to_numpy(dtype=np.float32)
y_raw = data.target.to_numpy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

print('X', X.shape, ' y', y.shape)
print('features:', list(data.feature_names))
print('classes :', list(label_encoder.classes_))
assert X.shape[1] == 16, f'expected 16 features, got {X.shape[1]}'
assert len(label_encoder.classes_) == CONFIG['num_classes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y)

# Standardize on train statistics only. The scaler is part of the model: the hardware
# encoder's thresholds live in this scaled space, so it gets saved with the checkpoint.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train).long()
y_test_t = torch.from_numpy(y_test).long()
print('train', X_train_t.shape, ' test', X_test_t.shape)

In [ ]:
# ---- thermometer binarization ----
# Pure PyTorch upstream, so this part would run fine locally too. Thresholds are fitted on
# TRAIN ONLY and must be saved -- Thermometer is not an nn.Module, so they are NOT in the
# state_dict, and without them the exported hardware encoder is undefined.
import torch_dwn as dwn

THERMOMETERS = {
    'plain': dwn.Thermometer,
    'gaussian': dwn.GaussianThermometer,
    'distributive': dwn.DistributiveThermometer,
}

thermometer = THERMOMETERS[CONFIG['thermometer']](CONFIG['thermometer_bits']).fit(X_train_t)

xb_train = thermometer.binarize(X_train_t).flatten(start_dim=1)
xb_test = thermometer.binarize(X_test_t).flatten(start_dim=1)

print('thresholds shape:', tuple(thermometer.thresholds.shape), '(features x bits)')
print('binarized train :', tuple(xb_train.shape))
print('input width     :', xb_train.size(1), 'bits')
assert xb_train.size(1) == 16 * CONFIG['thermometer_bits']

In [ ]:
# ---- build the model ----
from torch import nn

layers, in_size = [], xb_train.size(1)
for i, width in enumerate(CONFIG['layers']):
    layers.append(dwn.LUTLayer(
        in_size, width, n=CONFIG['n'],
        mapping=CONFIG['first_layer_mapping'] if i == 0 else 'random',
    ))
    in_size = width
layers.append(dwn.GroupSum(k=CONFIG['num_classes'], tau=CONFIG['tau']))

model = nn.Sequential(*layers).cuda()
print(model)
print()
for name, p in model.named_parameters():
    print(f'{name:30s} {str(tuple(p.shape)):20s} trainable={p.requires_grad}')

In [ ]:
# ---- train ----
import time
from torch.nn.functional import cross_entropy

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=CONFIG['lr_step'], gamma=CONFIG['lr_gamma'])

xb_train_g, y_train_g = xb_train.cuda(), y_train_t.cuda()
xb_test_g, y_test_g = xb_test.cuda(), y_test_t.cuda()


def evaluate():
    model.eval()
    with torch.no_grad():
        pred = model(xb_test_g).argmax(dim=1)
        return (pred == y_test_g).float().mean().item()


n_samples = xb_train_g.size(0)
best_acc, history = 0.0, []
t0 = time.time()

for epoch in range(CONFIG['epochs']):
    model.train()
    perm = torch.randperm(n_samples, device='cuda')
    for i in range(0, n_samples, CONFIG['batch_size']):
        idx = perm[i:i + CONFIG['batch_size']]
        optimizer.zero_grad()
        loss = cross_entropy(model(xb_train_g[idx]), y_train_g[idx])
        loss.backward()
        optimizer.step()
    scheduler.step()

    acc = evaluate()
    best_acc = max(best_acc, acc)
    history.append(acc)
    print(f'epoch {epoch+1:3d}/{CONFIG["epochs"]}  loss {loss.item():.4f}  test acc {acc:.4f}')

print()
print(f'final {history[-1]:.4f}   best {best_acc:.4f}   {time.time()-t0:.0f}s')
print()
print('Reference JSC numbers (project-brief.md §8, xcvu9p out-of-context):')
print('  DWN sm-10  71.2%    DWN sm-50  74.0%    DWN large  76.3%    hls4ml  76.2%')
print('Landing near the config-appropriate number means the training setup is sound.')

In [ ]:
# ---- inspect what the exporter will actually have to read ----
# This is the checkpoint-format reconnaissance called for in project-brief.md §12 risk #5.
# Do not guess this format later -- record what you see here.
print('=== parameters ===')
for name, p in model.state_dict().items():
    print(f'{name:34s} {str(tuple(p.shape)):18s} {p.dtype}')

print()
print('=== LUT tables ===')
lut0 = model[0].luts
print('shape', tuple(lut0.shape), '= (output_size, 2**n)')
print('range [{:.3f}, {:.3f}] -- clamped to [-1,1] during training'.format(
    lut0.min().item(), lut0.max().item()))
print('ONLY THE SIGN MATTERS at inference: the exporter thresholds these at 0.')
print('first node, first 16 entries as bits:', (lut0[0][:16] > 0).int().tolist())

print()
print('=== mapping (which input bits each node reads) ===')
for i, layer in enumerate(model[:-1]):
    m = layer.mapping
    if isinstance(m, dwn.LearnableMapping):
        print(f'layer {i}: LearnableMapping -- take argmax over its weights to get fixed wiring')
        print('         weight shape', tuple(m.weights.shape))
    else:
        print(f'layer {i}: fixed tensor {tuple(m.shape)} {m.dtype} = (output_size, n)')
        print('         node 0 reads input bits', m[0].tolist())

In [ ]:
# ---- save the checkpoint ----
# Everything the exporter needs, in one file. The thermometer thresholds and the scaler are
# NOT in the state_dict but ARE part of the model -- the hardware encoder is undefined
# without them.
import json, numpy as np

OUT = '/kaggle/working/dwn_jsc_checkpoint.pt'

torch.save({
    'config': CONFIG,
    'pinned_commit': PINNED_COMMIT,
    'state_dict': model.state_dict(),
    'thermometer': {
        'kind': CONFIG['thermometer'],
        'num_bits': CONFIG['thermometer_bits'],
        'thresholds': thermometer.thresholds.cpu(),
    },
    'scaler': {
        'mean': torch.from_numpy(scaler.mean_.astype(np.float32)),
        'scale': torch.from_numpy(scaler.scale_.astype(np.float32)),
    },
    'classes': list(label_encoder.classes_),
    'feature_names': list(data.feature_names),
    'results': {'final_acc': history[-1], 'best_acc': best_acc, 'history': history},
    'torch_version': torch.__version__,
}, OUT)

print('wrote', OUT)

# A few test vectors for the golden model / Gate 1 testbench, saved as plain arrays.
np.savez_compressed(
    '/kaggle/working/dwn_jsc_testvectors.npz',
    x_binarized=xb_test[:1000].numpy().astype(np.uint8),
    x_raw=X_test[:1000],
    y=y_test[:1000],
    pred=model(xb_test_g[:1000]).argmax(dim=1).cpu().numpy(),
)
print('wrote /kaggle/working/dwn_jsc_testvectors.npz')
print()
print('Download both from the Output panel on the right, then commit them (or their')
print('metadata) into the dwn-fpga repo. The .npz is what Gate 1 checks the RTL against.')

## After this runs

1. Download `dwn_jsc_checkpoint.pt` and `dwn_jsc_testvectors.npz` from the Output panel.
2. Record the accuracy against the §8 reference numbers. If it is far off, the training setup
   is wrong and the exporter would faithfully export a bad model.
3. Write down the checkpoint structure from the inspection cell into the exporter's design
   notes — that closes project-brief.md §12 risk #5.

**Speeding up Phase 2:** the nvcc build is 2–5 minutes every fresh session. Before the sweep,
build the wheel once and save it as a Kaggle Dataset, then install from that instead of
recompiling. Not worth doing yet.